<a href="https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%201/ISBA2411_Lecture2_Demo_Bigram_Babble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a Bigram Model, and Watch It Babble
### ISBA 2411 · Natural Language Processing · Live demo (Lecture 2, slide 22)

In Exercise 2 you built a bigram model **by hand** on five sentences. Now we build the same thing **in code**, watch it generate, then feed it real text and watch it babble.

**A bigram model plays one game:** given the word I just said, which word usually comes next? It looks at the previous word and nothing else. That short memory is the whole story tonight.

No installs, no API key. Just `Runtime → Run all`.

## 1 · The model in ten lines

Two helpers: one counts which word follows which, the other turns those counts into probabilities.

In [ ]:
import re, random
from collections import defaultdict, Counter

def tokenize(text):
    "Lowercase and split into word tokens."
    return re.findall(r"[a-z']+", text.lower())

def build_bigrams(tokens):
    "model[w1][w2] = how many times w2 followed w1."
    model = defaultdict(Counter)
    for w1, w2 in zip(tokens, tokens[1:]):
        model[w1][w2] += 1
    return model

def probabilities(model, word):
    "Turn the counts after `word` into fractions that sum to 1."
    counts = model[word]
    total = sum(counts.values())
    return {w: c / total for w, c in counts.items()}

print("Model ready.")

## 2 · The Exercise 2 corpus (does the code match your hand calc?)

These are the five sentences from the worksheet. Let's confirm the code produces the same numbers you computed for the word **"the"**.

In [ ]:
corpus = """
the small cat sat quietly
the small cat sat on the mat
the big dog sat on the rug
the small cat ran very fast
the big dog ran on the grass
"""

tokens = tokenize(corpus)
model  = build_bigrams(tokens)

print('After "the":')
for word, p in sorted(probabilities(model, "the").items(), key=lambda x: -x[1]):
    bar = "#" * round(p * 24)
    print(f'   the -> {word:8s} {p:5.3f}  {bar}')

You should see exactly the hand-calc numbers: `small 0.375`, `big 0.250`, and `mat / rug / grass 0.125` each. Same model, now in code.

## 3 · Generate, greedily — and watch it loop

The greedy rule: always take the single most likely next word, then repeat from *that* word.

In [ ]:
def generate_greedy(model, start, length=14):
    word = start
    out  = [word]
    for _ in range(length - 1):
        if not model[word]:          # dead end, no follower
            break
        word = model[word].most_common(1)[0][0]
        out.append(word)
    return " ".join(out)

print(generate_greedy(model, "the", 16))

It gets stuck: **"the small cat sat on the small cat sat on ..."** forever.

Why? After five words it lands back on **"the,"** and because the model only remembers the *previous* word, it makes the identical choice again. Goldfish memory. That is the Markov assumption, and it is exactly the wall we hit next in the lecture.

## 4 · Generate by sampling — same model, different behavior

Instead of always taking the top word, *roll a weighted die*: pick the next word in proportion to its probability. Now the model can branch, and can even reach an ending word and stop.

In [ ]:
def generate_sample(model, start, length=20, seed=None):
    if seed is not None:
        random.seed(seed)
    word = start
    out  = [word]
    for _ in range(length - 1):
        followers = model[word]
        if not followers:
            break
        choices, weights = zip(*followers.items())
        word = random.choices(choices, weights=weights)[0]
        out.append(word)
    return " ".join(out)

for s in range(5):
    print(generate_sample(model, "the", 12, seed=s))

Run it a few times. Some sentences wander (`the big dog ran on the ...`), some terminate cleanly (`... cat ran very fast`). **Greedy loops; sampling explores.** Same probabilities, different way of choosing.

## 5 · Now feed it real text, and watch it babble

Five sentences cannot babble interestingly. Let's train the same model on a whole book (*Alice in Wonderland*, public domain) and sample from it.

In [ ]:
import urllib.request

FALLBACK = ("the king said gravely it is a very good practice to begin at the beginning "
            "and go on till you come to the end then stop the rabbit took a watch out of "
            "its waistcoat pocket and looked at it and hurried on alice thought this very "
            "curious but the whole thing was so out of the way that she ran across the field") * 6

def load_book(url, fallback=FALLBACK):
    try:
        with urllib.request.urlopen(url, timeout=15) as r:
            text = r.read().decode("utf-8", errors="ignore")
        # strip Project Gutenberg header/footer if present
        if "*** START" in text:
            text = text.split("*** START", 1)[1]
        if "*** END" in text:
            text = text.split("*** END", 1)[0]
        return text
    except Exception as e:
        print("Download failed, using built-in fallback text. Reason:", e)
        return fallback

book_text   = load_book("https://www.gutenberg.org/files/11/11-0.txt")
book_tokens = tokenize(book_text)
book_model  = build_bigrams(book_tokens)
print(f"Trained on {len(book_tokens):,} words, {len(book_model):,} unique words.")

In [ ]:
# Babble! Start from a common word and sample.
for start in ["the", "she", "alice"]:
    print(f'[{start}] ', generate_sample(book_model, start, 25))
    print()

It produces fluent-sounding fragments that mean nothing. That is the headline of the whole lecture: **a model can sound like English without understanding a single word.**

## 6 · Give it more memory: a trigram (optional payoff)

A bigram remembers one word. A **trigram** remembers two. Watch the babble get noticeably more coherent, just by widening the window. This is the on-ramp to why the rest of the course chases longer context and real meaning.

In [ ]:
def build_trigrams(tokens):
    model = defaultdict(Counter)
    for w1, w2, w3 in zip(tokens, tokens[1:], tokens[2:]):
        model[(w1, w2)][w3] += 1
    return model

def generate_trigram(model, start_pair, length=30, seed=None):
    if seed is not None:
        random.seed(seed)
    w1, w2 = start_pair
    out = [w1, w2]
    for _ in range(length - 2):
        followers = model[(w1, w2)]
        if not followers:
            break
        choices, weights = zip(*followers.items())
        nxt = random.choices(choices, weights=weights)[0]
        out.append(nxt)
        w1, w2 = w2, nxt
    return " ".join(out)

tri = build_trigrams(book_tokens)
for seed in range(3):
    print(generate_trigram(tri, ("the", "queen"), 28, seed=seed))
    print()

More memory, better babble — but still zero understanding. To go further you cannot just keep widening the window (it explodes, the wall from slide 16). You need a model that captures *meaning*, which is where Week 2 and word embeddings begin.

## 7 · Your turn

- Change the **start word** in the babble cell. Try a rare word vs. a common one.
- Switch a generator from `generate_sample` to `generate_greedy` on the book model. Does the big model loop too?
- Train on a different public-domain book: swap the Gutenberg URL (find one at gutenberg.org, use the `.txt` link).
- Compare bigram vs. trigram output side by side. Where does the extra word of memory help most?

**One-line takeaway for the class:** counting word pairs gets you fluent nonsense. Understanding takes more than the previous word.